In [11]:
!pip install opencv-python ultralytics numpy

Basic Solution

In [ ]:
import cv2
from ultralytics import YOLO

def theSkynet(input_video_path, output_video_path):
    """
    Runs YOLOv8 detection and tracking on a video, saving the annotated output.
    """

    # 1. Load the YOLOv8 model. 
    # 'yolov8n.pt' is the nano model (fastest). 
    # For better accuracy with occlusions, you can change this to 'yolov8m.pt' or 'yolov8x.pt'.
    # The model will download automatically on the first run.
    print("Loading YOLOv8 model...")
    model = YOLO('yolov8n.pt')

    # 2. Open the input video
    cap = cv2.VideoCapture(input_video_path)
    if not cap.isOpened():
        print(f"Error: Cannot open video file {input_video_path}. Please check the path.")
        return

    # 3. Retrieve video properties to format the output video correctly
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    # Define the codec and create a VideoWriter object
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

    print(f"Processing video: {input_video_path}")
    print("Tracking subjects and generating annotated frames. This may take a while...")

    # 4. Run the tracking algorithm.
    # - source: the input video
    # - persist=True: forces the model to remember IDs across frames (Multi-Object Tracking)
    # - classes=[0]: filters detections to ONLY include class 0 ('person' in COCO dataset)
    # - stream=True: uses a generator to process one frame at a time (memory efficient)
    # - tracker="bytetrack.yaml": Explicitly uses ByteTrack (highly effective for occlusions)
    results = model.track(
        source=input_video_path, 
        persist=True, 
        classes=[0], 
        stream=True,
        tracker="bytetrack.yaml" 
    )

    # 5. Iterate through the processed frames and write them to the output video
    for result in results:
        # result.plot() automatically draws bounding boxes, labels, and tracking IDs
        annotated_frame = result.plot()
        
        # Write the annotated frame to our output video file
        out.write(annotated_frame)
        
        cv2.imshow("Tracking View", annotated_frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            print("Processing interrupted")
            break

    # 6. Clean up resources
    cap.release()
    out.release()
    cv2.destroyAllWindows()
    print(f"\nSuccess! Annotated video saved to: {output_video_path}")

if __name__ == "__main__":
    # Put your downloaded public sports video in the same folder and update the name here:
    input_file = "vid.mp4" 
    
    # The script will generate this file with bounding boxes and IDs:
    output_file = "output_tracked.mp4"
    
    theSkynet(input_file,output_file)

##Advance Tracking 

#along with number of id tracking 

#speed of players within the range of 30fps tracking

#trajectory trail

In [5]:
import cv2
import numpy as np
from collections import defaultdict
from ultralytics import YOLO

def theSkynet(input_video_path, output_video_path):
    print("Loading YOLOv8 model for advanced tracking...")
    model = YOLO('yolov8n.pt')

    cap = cv2.VideoCapture(input_video_path)
    if not cap.isOpened():
        print(f"Error: Cannot open {input_video_path}")
        return

    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

    # --- ENHANCEMENT DATA STRUCTURES ---
    # Dictionary to store the past N center points of each active ID for trajectories
    track_history = defaultdict(lambda: []) 
    # Set to store every unique ID that has ever appeared
    total_unique_ids = set()                

    print("Processing frames and calculating movement statistics...")

    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break

        # Run tracking (verbose=False keeps the console clean)
        results = model.track(frame, persist=True, classes=[0], verbose=False)
        
        # Get the base annotated frame (bounding boxes and IDs)
        annotated_frame = results[0].plot()
        current_active_count = 0
        
        # Check if any objects are detected in the current frame
        if results[0].boxes.id is not None:
            # Extract bounding boxes (x-center, y-center, width, height) and IDs
            boxes = results[0].boxes.xywh.cpu()
            track_ids = results[0].boxes.id.int().cpu().tolist()
            
            current_active_count = len(track_ids)
            
            for box, track_id in zip(boxes, track_ids):
                # Update total cumulative count
                total_unique_ids.add(track_id)
                
                # Calculate center coordinates
                x, y, w, h = box
                center = (float(x), float(y))
                
                # --- 1. TRAJECTORY VISUALIZATION ---
                track_history[track_id].append(center)
                if len(track_history[track_id]) > 30:  # Keep the tail length to 30 frames
                    track_history[track_id].pop(0)

                #print(track_history)
                # Draw the trajectory trail
                points = np.hstack(track_history[track_id]).astype(np.int32).reshape((-1, 1, 2))
                cv2.polylines(annotated_frame, [points], isClosed=False, color=(0, 255, 255), thickness=2)
                
                # --- 2. MOVEMENT STATISTICS (PIXEL SPEED) ---
                # Calculate Euclidean distance between the current frame and the previous frame
                if len(track_history[track_id]) >= 2:
                    dx = track_history[track_id][-1][0] - track_history[track_id][-2][0]
                    dy = track_history[track_id][-1][1] - track_history[track_id][-2][1]
                    
                    # Pixels per frame * frames per second = Pixels per second
                    speed_px_per_sec = np.sqrt(dx**2 + dy**2) * fps 
                    
                    # Display speed above the bounding box
                    cv2.putText(annotated_frame, f"{speed_px_per_sec:.0f} px/s", 
                                (int(x - w/2), int(y - h/2)), 
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

        # --- 3. OBJECT COUNT OVER TIME (UI DASHBOARD) ---
        # Draw a semi-transparent black background for readability
        overlay = annotated_frame.copy()
        cv2.rectangle(overlay, (10, 10), (350, 90), (0, 0, 0), -1)
        cv2.addWeighted(overlay, 0.6, annotated_frame, 0.4, 0, annotated_frame)
        
        # Add the text metrics
        cv2.putText(annotated_frame, f"Active on Screen: {current_active_count}", (20, 40), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        cv2.putText(annotated_frame, f"Total Unique Subjects: {len(total_unique_ids)}", (20, 75), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

        out.write(annotated_frame)
        
        cv2.imshow("Advanced Tracking", annotated_frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    out.release()
    cv2.destroyAllWindows()
    print(f"Finished! Saved to {output_video_path}")

if __name__ == "__main__":
    input_vid="vid.mp4"
    output_vid="advanced_output.mp4"
    theSkynet(input_vid,output_vid)

Loading YOLOv8 model for advanced tracking...
Processing frames and calculating movement statistics...
Finished! Saved to advanced_output.mp4


print("""Detector Used: YOLOv8 (You Only Look Once, version 8). It is an anchor-free object detection model that provides state-of-the-art accuracy and speed.

Tracker Used: ByteTrack (integrated into Ultralytics).

Why this combination? ByteTrack excels at handling occlusion and similar-looking subjects because, unlike traditional trackers that discard low-confidence detections, ByteTrack associates even low-confidence detection boxes with tracklets. This means if a player is partially occluded (lowering the detection confidence), ByteTrack will still maintain their ID rather than dropping them.

How ID consistency is maintained: The persist=True argument maps bounding boxes across sequential frames using intersection-over-union (IoU) algorithms combined with Kalman filters to predict where a moving player will be in the next frame.

Dataset Filter: The parameter classes=[0] forces the model to ignore balls, cars, or background objects and strictly focus on tracking people (athletes/players), which perfectly suits the sports requirement.""")

#MultiThreaded Solution

In [ ]:
import cv2
import threading
import queue
import time
import numpy as np
from collections import defaultdict
from ultralytics import YOLO

class ThreadedTracker:
    def __init__(self, input_path, output_path, model_name='yolov8n.pt'):
        self.input_path = input_path
        self.output_path = output_path
        self.model = YOLO(model_name)
        
        # Queues for inter-thread communication
        self.frame_queue = queue.Queue(maxsize=30)  # Buffer for raw frames
        self.output_queue = queue.Queue(maxsize=30) # Buffer for annotated frames
        
        self.stop_event = threading.Event()
        
        # Tracking data
        self.track_history = defaultdict(lambda: [])
        self.total_unique_ids = set()

    def video_reader(self):
        """Thread 1: Reads frames from disk and pushes to queue."""
        cap = cv2.VideoCapture(self.input_path)
        if not cap.isOpened():
            print(f"Error: Cannot open {input_video_path}")
            print("Press Ctrl+C to stop the process")
            
        while not self.stop_event.is_set():
            ret, frame = cap.read()
            if not ret:
                self.stop_event.set()
                break
            
            # If queue is full, this will block until space is available
            self.frame_queue.put(frame)
            
        cap.release()
        print("[Reader] Finished reading file.")

    def video_processor(self, fps):
        """Thread 2: The Core Engine. Handles YOLO, Tracking, and Analytics."""
        while not self.stop_event.is_set() or not self.frame_queue.empty():
            try:
                frame = self.frame_queue.get(timeout=2)
            except queue.Empty:
                continue

            # Run Tracking
            results = self.model.track(frame, persist=True, classes=[0], verbose=False, tracker="bytetrack.yaml")
            annotated_frame = results[0].plot()

            if results[0].boxes.id is not None:
                boxes = results[0].boxes.xywh.cpu()
                track_ids = results[0].boxes.id.int().cpu().tolist()

                for box, track_id in zip(boxes, track_ids):
                    self.total_unique_ids.add(track_id)
                    x, y, w, h = box
                    center = (float(x), float(y))
                    
                    # Trajectory tail logic
                    self.track_history[track_id].append(center)
                    if len(self.track_history[track_id]) > 30:
                        self.track_history[track_id].pop(0)

                    # Draw Tail
                    points = np.hstack(self.track_history[track_id]).astype(np.int32).reshape((-1, 1, 2))
                    cv2.polylines(annotated_frame, [points], isClosed=False, color=(0, 255, 255), thickness=2)

                    if len(track_history[track_id]) >= 2:
                    dx = track_history[track_id][-1][0] - track_history[track_id][-2][0]
                    dy = track_history[track_id][-1][1] - track_history[track_id][-2][1]
                    
                    # Pixels per frame * frames per second = Pixels per second
                    speed_px_per_sec = np.sqrt(dx**2 + dy**2) * fps 
                    
                    # Display speed above the bounding box
                    cv2.putText(annotated_frame, f"{speed_px_per_sec:.0f} px/s", 
                                (int(x - w/2), int(y - h/2)), 
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

            # UI Overlay
            cv2.putText(annotated_frame, f"IDs Tracked: {len(self.total_unique_ids)}", (20, 40), 
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

            self.output_queue.put(annotated_frame)
            self.frame_queue.task_done()

    def video_writer(self, fps, width, height):
        """Thread 3: Encodes and writes processed frames to disk."""
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(self.output_path, fourcc, fps, (width, height))
        
        while not self.stop_event.is_set() or not self.output_queue.empty():
            try:
                frame = self.output_queue.get(timeout=2)
                out.write(frame)
                self.output_queue.task_done()
            except queue.Empty:
                continue
                
        out.release()
        print("[Writer] Finished writing file.")

    def start(self):
        # Initial capture to get video metadata
        cap = cv2.VideoCapture(self.input_path)
        fps = int(cap.get(cv2.CAP_PROP_FPS))
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        cap.release()

        # Define and start threads
        t1 = threading.Thread(target=self.video_reader)
        t2 = threading.Thread(target=self.video_processor, args=(fps,))
        t3 = threading.Thread(target=self.video_writer, args=(fps, width, height))

        start_time = time.time()
        t1.start()
        t2.start()
        t3.start()

        t1.join()
        t2.join()
        t3.join()
        
        print(f"Total Processing Time: {time.time() - start_time:.2f} seconds")

if __name__ == "__main__":
    tracker = ThreadedTracker("vid.mp4", "threaded_output.mp4")
    tracker.start()